In [1]:
import json
from crewai import Agent, Task, Crew, Process, LLM

# TUTORING FLOW

input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph

In [2]:
# --- 1. Helper Function: File Reading ---
def read_text_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file {file_path}: {str(e)}"

# --- 2. Konfigurasi LLM ---
llm = LLM(
    model="ollama/llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.1  # Low temp agar output JSON konsisten
)

# Load Knowledge Base
# 1. Panduan Pseudocode (untuk Style Checker)
pseudocode_path = '/home/ilham/Documents/python/crewai-vs-langgraph/doc/Pseudocode dan Golang Dasar.md'
pseudocode_knowledge = read_text_file(pseudocode_path)

# 2. Daftar Miskonsepsi (untuk Logic Checker)
misconceptions_path = '/home/ilham/Documents/python/crewai-vs-langgraph/doc/List of misconceptions.md'
misconceptions_knowledge = read_text_file(misconceptions_path)

# --- 3. Definisi Agen (Sub-Agents & Supervisor) ---

style_checker_agent = Agent(
    role="Style & Syntax Auditor",
    goal="Mendeteksi kesalahan format, deklarasi variabel, dan kepatuhan standar penulisan pseudocode.",
    backstory=(
        "Anda adalah asisten dosen yang sangat teliti terhadap detail penulisan.\n"
        "HANYA cek: Kamus variabel (lengkap/tidak), Tipe data (sesuai/tidak), "
        "Konsistensi nama variabel (typo antara kamus dan algoritma), "
        "Struktur dasar (program/endprogram, if/endif).\n\n"
        "JANGAN analisis logika (kondisi benar/salah, output sesuai/tidak). Itu bukan tugas Anda.\n"
        "Contoh yang BUKAN kesalahan style: 'if (x mod 2 = 1)' vs 'if (x mod 2 = 0)' - ini logika, bukan style.\n\n"
        f"Acuan Standar:\n{pseudocode_knowledge}"
    ),
    llm=llm,
    verbose=True
)

logic_checker_agent = Agent(
    role="Algorithmic Logic Analyst",
    goal="Membandingkan logika siswa dengan solusi referensi untuk menemukan kesalahan alur dan miskonsepsi.",
    backstory=(
        "Anda adalah ahli algoritma.\n"
        "Bandingkan 'pseudocode siswa' dengan 'context solution'.\n\n"
        "CEK SPESIFIK:\n"
        "1. Kondisi If/Else: Apakah kondisi TERBALIK atau output TERBALIK?\n"
        "2. Perulangan: Start, stop, increment benar?\n"
        "3. Operasi Matematika: Rumus benar? Edge cases tertangani?\n\n"
        "IDENTIFIKASI MISKONSEPSI berdasarkan daftar berikut, HANYA jika benar-benar cocok:\n"
        f"{misconceptions_knowledge}\n\n"
        "ABAIKAN:\n"
        "- Sintaks `=` vs `==` (itu detail implementasi, bukan logika)\n"
        "- Typo nama variabel (tugas Style Auditor)\n"
        "- Gunakan `print` vs `output` (tugas Style Auditor)\n\n"
        "FOKUS: Apakah ALUR dan OUTPUT program benar secara fungsional?"
    ),
    llm=llm,
    verbose=True
)

scoring_supervisor_agent = Agent(
    role="Scoring Supervisor",
    goal="Mengkonsolidasi laporan, menghitung skor berdasarkan rubrik, dan menghasilkan JSON final.",
    backstory=(
        "Anda adalah Kepala Penilai.\n"
        "Terima laporan dari Style Auditor dan Logic Analyst.\n\n"
        "TUGAS:\n"
        "1. Hitung pengurangan poin berdasarkan 'general_rubrication'.\n"
        "2. Tentukan 'correct': true jika program menghasilkan output benar untuk semua kasus, false jika ada kesalahan.\n"
        "3. Rangkum kesalahan utama (summary).\n"
        "4. List miskonsepsi HANYA yang disebutkan Logic Analyst (jika ada).\n"
        "5. Salin PSEUDOCODE ASLI siswa (dari input), BUKAN pseudocode yang sudah diperbaiki.\n\n"
        "WAJIB output JSON murni tanpa markdown block."
    ),
    llm=llm,
    verbose=True
)


# --- 4. Definisi Task ---

# Task 1: Cek Style
style_task = Task(
    description=(
        "Analisis HANYA aspek styling dan sintaks dari pseudocode siswa.\n\n"
        "CEK:\n"
        "- Apakah semua variabel yang digunakan sudah dideklarasikan di kamus?\n"
        "- Apakah tipe data variabel sesuai (integer, string, boolean, dll)?\n"
        "- Apakah nama variabel konsisten antara kamus dan algoritma (typo)?\n"
        "- Apakah struktur dasar lengkap (program...endprogram, if...endif)?\n\n"
        "JANGAN CEK logika (kondisi benar/salah, output sesuai/tidak).\n\n"
        "Pseudocode Siswa:\n{pseudocode}"
    ),
    expected_output="List kesalahan styling dan sintaks saja, tanpa analisis logika.",
    agent=style_checker_agent
)

logic_task = Task(
    description=(
        "Analisis logika pseudocode siswa untuk problem ini.\n"
        "Bandingkan dengan context_solution.\n\n"
        "Problem: {problem}\n"
        "Context Solution: {context_solution}\n"
        "Pseudocode Siswa: {pseudocode}\n\n"
        "ANALISIS:\n"
        "1. Apakah kondisi If/Else menghasilkan output yang BENAR?\n"
        "   - Jika kondisi untuk 'Genap' salah, maka output akan terbalik.\n"
        "2. Apakah ada kesalahan perhitungan matematika?\n"
        "3. Apakah edge cases tertangani (misal: N=0, N negatif)?\n\n"
        "MISKONSEPSI:\n"
        "Sebutkan HANYA jika siswa menunjukkan pola miskonsepsi dari daftar.\n"
        "JANGAN paksa miskonsepsi jika tidak ada.\n\n"
        "KATEGORIKAN kesalahan:\n"
        "- FATAL: Program tidak berfungsi sama sekali\n"
        "- MAYOR: Sebagian besar kasus menghasilkan output salah\n"
        "- MINOR: Hanya edge case tertentu yang salah"
    ),
    expected_output="Analisis logika, kesalahan mayor/minor, dan miskonsepsi (jika ada).",
    agent=logic_checker_agent
)

scoring_task = Task(
    description=(
        "Berdasarkan laporan Style Auditor dan Logic Analyst, buat penilaian final.\n\n"
        "Rubrik: {general_rubrication}\n\n"
        "LANGKAH PENILAIAN:\n"
        "1. Mulai dari 100 poin\n"
        "2. Kurangi berdasarkan kesalahan yang ditemukan\n"
        "3. Jika logika inti terbalik (genap jadi ganjil), kurangi 40 poin (kesalahan mayor)\n"
        "4. Typo variabel: -5 poin per typo\n"
        "5. Format output salah (print vs output): -5 poin\n\n"
        "OUTPUT JSON (HANYA JSON, tanpa ```json```):\n"
        "{\n"
        '  "score": 60,\n'
        '  "correct": false,\n'
        '  "summary": "Logika terbalik: kondisi untuk genap menghasilkan output ganjil, dan sebaliknya.",\n'
        '  "Misconceptions": ["Nama miskonsepsi dari Logic Analyst (jika ada)"],\n'
        '  "pseudocode": "<SALIN PSEUDOCODE ASLI SISWA DI SINI, BUKAN YANG SUDAH DIPERBAIKI>"\n'
        "}\n\n"
        "PENTING: Field 'pseudocode' harus berisi PSEUDOCODE ASLI dari input, bukan versi yang sudah diperbaiki!"
    ),
    expected_output="Valid JSON string dengan score, correct, summary, misconceptions, dan pseudocode ASLI.",
    agent=scoring_supervisor_agent,
    context=[style_task, logic_task]
)

# --- 5. Definisi Crew ---
grading_crew = Crew(
    agents=[style_checker_agent, logic_checker_agent, scoring_supervisor_agent],
    tasks=[style_task, logic_task, scoring_task],
    process=Process.sequential # Sequential agar Supervisor mendapat konteks sub-agent
)

# --- 6. Data Input & Eksekusi ---

# Contoh Data Input
input_data = {
    'problem': "Buatlah algoritma untuk menentukan apakah sebuah bilangan N adalah Ganjil atau Genap. Output 'Ganjil' atau 'Genap'.",
    
    'context_solution': """
        Program GanjilGenap
        kamus
            N : integer
        algoritma
            input(N)
            if (N mod 2 == 0) then
                output("Genap")
            else
                output("Ganjil")
            endif
        endprogram
    """,
    
    'pseudocode': """
        Program CekBilangan
        kamus
            bil : integer
        algoritma
            input(bil)
            if (bil mod 2 = 1) then
                print("Genap")  
            else
                print("Ganjil")
        endprogram
    """,
    
    'general_rubrication': """
        Start Score: 100.
        Pengurangan:
        1. Kesalahan Kritis:
        - Logika inti sepenuhnya salah, menghasilkan output tidak relevan: -50 Poin
        - Program tidak menyelesaikan masalah sama sekali (ada usaha): -90 Poin
        - Lembar jawaban kosong: -50 Poin

        2. Kesalahan Logika Mayor:
        - Gagal menangani salah satu kondisi utama: -40 Poin
        - Perhitungan matematis utama tidak akurat: -40 Poin
        - Variabel tidak tertulis pada kamus: -25 Poin

        3. Kesalahan Logika Minor & Struktur:
        - Gagal menangani kasus khusus (edge case): -25 Poin
        - Tidak menggunakan tipe bentukan (jika diwajibkan): -25 Poin
        - Alur program tidak efisien/berbelit: -15 Poin

        4. Kesalahan Kelengkapan:
        - Tipe data variabel tidak sesuai: -10 Poin
        - Penulisan variabel berbeda (typo) antara kamus & program: -5 Poin
        - Format output tidak sesuai: -5 Poin
    """
}

print("### MEMULAI PROSES PENILAIAN BERTINGKAT ###")
result = grading_crew.kickoff(inputs=input_data)

print("\n\n########################")
print("## HASIL JSON FINAL ##")
print("########################\n")

# Membersihkan output jika LLM menambahkan markdown block secara tidak sengaja
clean_result = str(result).replace("```json", "").replace("```", "").strip()

try:
    # Validasi apakah output benar-benar JSON
    json_output = json.loads(clean_result)
    print(json.dumps(json_output, indent=2))
except json.JSONDecodeError:
    print("Warning: Output Raw (Gagal Parsing JSON Murni):")
    print(clean_result)

### MEMULAI PROSES PENILAIAN BERTINGKAT ###


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Task: Analisis HANYA aspek styling dan sintaks dari pseudocode siswa.                                          │
│                                                                                                                 │
│  CEK:                                                                                                           │
│  - Apakah semua variabel yang digunakan sudah dideklarasikan di kamus?                                          │
│  - Apakah tipe data variabel sesuai (integer, string, boolean, dll)?                                            │
│  - Apakah nama variabel konsisten antara kamus dan algoritma (typo)?                                            │
│  - Apakah struktur dasar lengkap (program...endprogram, if...endif)?                                            │
│                                                                                                                 │
│  JANGAN CEK logika (kondisi benar/salah, output sesuai/tidak).                                                  │
│                                                                                                                 │
│  Pseudocode Siswa:                                                                                              │
│                                                                                                                 │
│          Program CekBilangan                                                                                    │
│          kamus                                                                                                  │
│              bil : integer                                                                                      │
│          algoritma                                                                                              │
│              input(bil)                                                                                         │
│              if (bil mod 2 = 1) then                                                                            │
│                  print("Genap")                                                                                 │
│              else                                                                                               │
│                  print("Ganjil")                                                                                │
│          endprogram                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Style & Syntax Auditor                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Variabel "bil" tidak dideklarasikan dengan tipe data yang tepat (integer).                                  │
│  2. Operator "=" digunakan pada pernyataan if, seharusnya menggunakan "==".                                     │
│  3. Tanda kurung pada pernyataan if tidak sesuai dengan aturan penulisan pseudocode.                            │
│  4. Pernyataan else tidak memiliki tanda kurung yang tepat.                                                     │
│  5. Fungsi "print" tidak dideklarasikan di kamus.                                                               │
│                                                                                                                 │
│  Note: Saya hanya menganalisis aspek styling dan sintaks, bukan logika atau kondisi benar/salah pada            │
│  pseudocode siswa.                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Algorithmic Logic Analyst                                                                               │
│                                                                                                                 │
│  Task: Analisis logika pseudocode siswa untuk problem ini.                                                      │
│  Bandingkan dengan context_solution.                                                                            │
│                                                                                                                 │
│  Problem: Buatlah algoritma untuk menentukan apakah sebuah bilangan N adalah Ganjil atau Genap. Output          │
│  'Ganjil' atau 'Genap'.                                                                                         │
│  Context Solution:                                                                                              │
│          Program GanjilGenap                                                                                    │
│          kamus                                                                                                  │
│              N : integer                                                                                        │
│          algoritma                                                                                              │
│              input(N)                                                                                           │
│              if (N mod 2 == 0) then                                                                             │
│                  output("Genap")                                                                                │
│              else                                                                                               │
│                  output("Ganjil")                                                                               │
│              endif                                                                                              │
│          endprogram                                                                                             │
│                                                                                                                 │
│  Pseudocode Siswa:                                                                                              │
│          Program CekBilangan                                                                                    │
│          kamus                                                                                                  │
│              bil : integer                                                                                      │
│          algoritma                                                                                              │
│              input(bil)                                                                                         │
│              if (bil mod 2 = 1) then                                                                            │
│                  print("Genap")                                                                                 │
│              else                                                                                               │
│                  print("Ganjil")                                                                                │
│          endprogram                                                                                             │
│                                                                                                                 │
│                                                        

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Algorithmic Logic Analyst                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  #### Analisis Logika:                                                                                          │
│                                                                                                                 │
│  1. Kondisi If/Else: Pada pseudocode siswa, kondisi if menggunakan operator "=" yang salah, seharusnya          │
│  menggunakan "==".                                                                                              │
│  2. Perhitungan Matematika: Tidak ada kesalahan perhitungan matematika pada pseudocode siswa.                   │
│  3. Edge Cases: Tidak ada edge cases yang tertangani secara spesifik pada pseudocode siswa.                     │
│                                                                                                                 │
│  #### Kesalahan:                                                                                                │
│                                                                                                                 │
│  - FATAL: Program tidak berfungsi sama sekali karena operator "=" digunakan pada pernyataan if dan tanda        │
│  kurung pada pernyataan if tidak sesuai dengan aturan penulisan pseudocode.                                     │
│  - MAYOR: Sebagian besar kasus menghasilkan output salah karena kondisi if menggunakan operator "=" yang        │
│  salah.                                                                                                         │
│                                                                                                                 │
│  #### Miskonsepsi:                                                                                              │
│                                                                                                                 │
│  Siswa menunjukkan pola miskonsepsi "Conditional statement without alternative as conditional loop" (IfWhile)   │
│  karena pernyataan else tidak memiliki tanda kurung yang tepat dan fungsi "print" tidak dideklarasikan di       │
│  kamus.                                                                                                         │
│                                                                                                                 │
│  #### Kategori Kesalahan:                                                                                       │
│                                                                                                                 │
│  - FATAL: Program tidak berfungsi sama sekali.                                                                  │
│  - MAYOR: Sebagian besar kasus menghasilkan output salah.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scoring Supervisor                                                                                      │
│                                                                                                                 │
│  Task: Berdasarkan laporan Style Auditor dan Logic Analyst, buat penilaian final.                               │
│                                                                                                                 │
│  Rubrik:                                                                                                        │
│          Start Score: 100.                                                                                      │
│          Pengurangan:                                                                                           │
│          1. Kesalahan Kritis:                                                                                   │
│          - Logika inti sepenuhnya salah, menghasilkan output tidak relevan: -50 Poin                            │
│          - Program tidak menyelesaikan masalah sama sekali (ada usaha): -90 Poin                                │
│          - Lembar jawaban kosong: -50 Poin                                                                      │
│                                                                                                                 │
│          2. Kesalahan Logika Mayor:                                                                             │
│          - Gagal menangani salah satu kondisi utama: -40 Poin                                                   │
│          - Perhitungan matematis utama tidak akurat: -40 Poin                                                   │
│          - Variabel tidak tertulis pada kamus: -25 Poin                                                         │
│                                                                                                                 │
│          3. Kesalahan Logika Minor & Struktur:                                                                  │
│          - Gagal menangani kasus khusus (edge case): -25 Poin                                                   │
│          - Tidak menggunakan tipe bentukan (jika diwajibkan): -25 Poin                                          │
│          - Alur program tidak efisien/berbelit: -15 Poin                                                        │
│                                                                                                                 │
│          4. Kesalahan Kelengkapan:                                                                              │
│          - Tipe data variabel tidak sesuai: -10 Poin                                                            │
│          - Penulisan variabel berbeda (typo) antara kamus & program: -5 Poin                                    │
│          - Format output tidak sesuai: -5 Poin                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  LANGKAH PENILAIAN:                                                                                             │
│  1. Mulai dari 100 poin                                                                                         │
│  2. Kurangi berdasarkan kesalahan yang ditemukan                                                                │
│  3. Jika logika inti terbalik (genap jadi ganjil), kurangi 40 poin (kesalahan mayor)                            │
│  4. Typo variabel: -5 poin per typo                    

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scoring Supervisor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "score": 10,                                                                                                 │
│    "correct": false,                                                                                            │
│    "summary": "Logika terbalik: kondisi untuk genap menghasilkan output ganjil, dan sebaliknya.",               │
│    "Misconceptions": ["Conditional statement without alternative as conditional loop"],                         │
│    "pseudocode": "if bil = 5 then print 'genap' else print 'ganjil'"                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



########################
## HASIL JSON FINAL ##
########################

{
  "score": 10,
  "correct": false,
  "summary": "Logika terbalik: kondisi untuk genap menghasilkan output ganjil, dan sebaliknya.",
  "Misconceptions": [
    "Conditional statement without alternative as conditional loop"
  ],
  "pseudocode": "if bil = 5 then print 'genap' else print 'ganjil'"
}


input:
- pseoudocode
- problem
- context solution
- general rubrication

agen:
- scoring supervisior
- sub-agent 1: style checker agent
- sub-agent 2: logic checker agent

output:
```json
{
  "score":"value_scorenya",
  "correct":"true/false",
  "summary":"some_summary_here",
  "Misconceptions":"misconceptionsnya apa",
  "pseudocode":"pseudocode_siswa"
}
```

Possible Prompt for Each Agent:
- zero-shot
- few-shot
- chain-of-thought

TO DO Voice:
- Mendesain multi-agent system dengan 3 agen di dalamnya yaitu Style Checker, Logic Checker, dan Scoring Supervisor. Gunakan iterasi ganjil dalam interaksi antar agent seperti 1,3,5,dst.
- Eksplorasi berbagai LLM (free resource and can be fine tuned) yang bisa digunakan ke dalam agent.
- Gunakan prompting zero-shot atau COT pada Style Checker dan Logic Checker (boleh prompting lain juga).
- Tech-Stacknya bisa pakai Langchain / Langgraph